In [3]:
import numpy as np                                        # 숫자 배열 계산
from sklearn.model_selection import train_test_split      # train/test 나누기
from sklearn.tree import DecisionTreeClassifier           # 결정트리 (분류)
from sklearn.ensemble import BaggingClassifier            # 배깅
from sklearn.ensemble import RandomForestClassifier       # 랜덤포레스트

In [ ]:
rng = np.random.default_rng(42)
N = 3000

X = rng.normal(size = (N,9))  # 피처 9개, 정규분포 랜덤값
score = X[:, 0] + 0.8*X[:,1] + 0.5*X[:,4]
Y = (score + rng.normal(scale=1.5, size = N) > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, Y, test_size=0.3, random_state=0)
print(X_tr.shape, X_te.shape)

(2100, 9) (900, 9)


In [5]:
depth =[1, 3, 5, 10, None]

for d in depth:
    tree = DecisionTreeClassifier(max_depth=d, random_state=0)
    tree.fit(X_tr,y_tr)

    tr = tree.score(X_tr, y_tr)
    te = tree.score(X_te, y_te)
    print(f"max_depth={str(d):>4}  train = {tr:.3f}  test={te:.3f}  격차={tr - te:.3f}")

max_depth=   1  train = 0.663  test=0.666  격차=-0.003
max_depth=   3  train = 0.730  test=0.701  격차=0.028
max_depth=   5  train = 0.765  test=0.706  격차=0.059
max_depth=  10  train = 0.925  test=0.674  격차=0.251
max_depth=None  train = 1.000  test=0.644  격차=0.356


In [ ]:
models ={
    "결정트리 1개" : DecisionTreeClassifier(random_state=0),

    "Bagging 300개": BaggingClassifier(
        DecisionTreeClassifier(),
        n_estimators=300, # 트리 개수
        random_state=0,
        n_jobs=-1 # cpu코어 몇 개 쓸지, -1은 전부 사용
    ),
    "RandomForest 300개": RandomForestClassifier(
        n_estimators=300,                                 # 트리 300개
        random_state=0,
        n_jobs=-1
    ), 
}

for name, model in models.items():
    tree.fit(X_tr,y_tr)

    tr = tree.score(X_tr, y_tr)
    te = tree.score(X_te, y_te)

    print(f"{name:<12} train={tr:.3f}  test={te:.3f}")

결정트리 1개      train=1.000  test=0.644
Bagging 300개 train=1.000  test=0.644
RandomForest 300개 train=1.000  test=0.644


In [13]:
P = 100
results = {k: [] for k in [1, 3, 10, 30, 100]} 

for seed in range(5):                                     # 시드 0~4, 5번 반복
    rng = np.random.default_rng(seed)                     # 시드마다 다른 데이터 생성
    X2 = rng.normal(size=(N, P))
    y2 = (X2[:, 0] + rng.normal(scale=0.7, size=N) > 0).astype(int)
    X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=seed)

    for k in [1, 3, 10, 30, 100]:                         # 안쪽 반복: max_features 바꾸기
        rf = RandomForestClassifier(
            n_estimators=100,                             # 시간 줄이려고 300 → 100
            max_features=k,
            random_state=seed,
            n_jobs=-1)
        rf.fit(X2_tr, y2_tr)
        te = rf.score(X2_te, y2_te)
        results[k].append(te)                             # 결과 저장
        print(f"seed={seed}  max_features={k:>3}  test={te:.3f}")
    print()                                               # 시드마다 한 줄 띄우기

print("=== k별 평균 ± 표준편차 ===")
for k, scores in results.items():
    print(f"max_features={k:>3}  평균={np.mean(scores):.3f}  ± {np.std(scores):.3f}")

seed=0  max_features=  1  test=0.631
seed=0  max_features=  3  test=0.744
seed=0  max_features= 10  test=0.764
seed=0  max_features= 30  test=0.771
seed=0  max_features=100  test=0.768

seed=1  max_features=  1  test=0.678
seed=1  max_features=  3  test=0.777
seed=1  max_features= 10  test=0.782
seed=1  max_features= 30  test=0.787
seed=1  max_features=100  test=0.787

seed=2  max_features=  1  test=0.646
seed=2  max_features=  3  test=0.789
seed=2  max_features= 10  test=0.816
seed=2  max_features= 30  test=0.812
seed=2  max_features=100  test=0.810

seed=3  max_features=  1  test=0.656
seed=3  max_features=  3  test=0.767
seed=3  max_features= 10  test=0.784
seed=3  max_features= 30  test=0.788
seed=3  max_features=100  test=0.786

seed=4  max_features=  1  test=0.666
seed=4  max_features=  3  test=0.811
seed=4  max_features= 10  test=0.813
seed=4  max_features= 30  test=0.818
seed=4  max_features=100  test=0.810

=== k별 평균 ± 표준편차 ===
max_features=  1  평균=0.655  ± 0.016
max_features=